# Process

##### Steps: 
1. read the column
2. define schema in StructType and StructField 
3. parse the schema using from_json function 
4. explode the parsed cloumn
5. combine the code
6. write it into silver schema

In [0]:

df = spark.read.table("ecommerce_analytics.bronze.sales")
df.display()

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import *

df = spark.read.table('ecommerce_analytics.bronze.sales')

product_schema = StructType([
    StructField('id', StringType(), True),
    StructField('name', StringType(), True),
    StructField('price', FloatType(), True),
    StructField('curr', StringType(), True),
    StructField('qty', IntegerType(), True),
    StructField('unit', StringType(), True)
])

''' # Another way to define schema of nested columns using _parse_datatype_string 
from pyspark.sql.types import _parse_datatype_string as p

schema = "curr int, id int, name string, price double, qty int, unit string"
schema = p(schema)
print(schema)
'''

df_sales = df.withColumn('product', from_json('product', product_schema)) \
                .select('customer_id', 'customer_name', 'product_name', 'order_date', 'product_category', 'product.*', 'total_price')
#df_sales.display()
df_sales.write.mode('overwrite').saveAsTable('ecommerce_analytics.silver.silver_sales')